# Project 3: Surgical Outcomes & Cardiovascular Risk
## NLP on Clinical Notes + Structured Data Fusion

---

**Objective:** Build an NLP pipeline to extract clinical comorbidities from surgical notes, then fuse those extracted features with structured heart disease data to predict cardiovascular risk in surgical patients.

**Core Question:** Does parsing clinical notes add predictive value beyond structured data alone?

**Datasets:**
- Anesthesia Dataset (300 patients) — surgical records with free-text clinical notes
- Heart Disease UCI (920 patients) — structured cardiovascular risk factors

---

### Executive Summary
*(Fill this in after completing the project)*

- **Problem:**
- **Approach:**
- **NLP Extraction Accuracy (F1):**
- **NLP Uplift (AUC improvement):**
- **Key Finding:**
- **Limitations:**

In [1]:
# Setup — run this cell first
import pandas as pd
import numpy as np
import re
import random
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# Reproducibility
random.seed(42)
np.random.seed(42)

print('Setup complete.')

Setup complete.


---
## Step 1 — Explore & Understand Both Datasets

Load both datasets. Examine shapes, types, missing values, and distributions. Pay special attention to the text columns — you'll discover something important.

In [4]:
# Load datasets
anes = pd.read_csv('Anesthesia_Dataset.csv')
heart = pd.read_csv('heart_disease_uci.csv')

print(f'Anesthesia: {anes.shape[0]} rows × {anes.shape[1]} columns')
print(f'Heart Disease: {heart.shape[0]} rows × {heart.shape[1]} columns')

Anesthesia: 300 rows × 12 columns
Heart Disease: 920 rows × 16 columns


In [5]:
# Anesthesia — overview
print('=== ANESTHESIA DATASET ===')
print()
print('Data types:')
print(anes.dtypes)
print()
print('Missing values:')
print(anes.isnull().sum()[anes.isnull().sum() > 0])
print()
anes.head()

=== ANESTHESIA DATASET ===

Data types:
PatientID              int64
Age                    int64
Gender                object
BMI                    int64
SurgeryType           object
SurgeryDuration       object
AnesthesiaType        object
PreoperativeNotes     object
PostoperativeNotes    object
PainLevel              int64
Complications         object
Outcome                int64
dtype: object

Missing values:
Complications    73
dtype: int64



,PatientID,Age,Gender,BMI,SurgeryType,SurgeryDuration,AnesthesiaType,PreoperativeNotes,PostoperativeNotes,PainLevel,Complications,Outcome
0,1,33,M,32,Neurological,217 min,Local,"Hypertension, diabetes","Minimal pain, no complications",7,NaN,0
1,2,33,M,23,Cardiovascular,181 min,Local,"Stable, no allergies","Minimal pain, no complications",7,"Nausea, mild bleeding",1
2,3,58,F,24,Orthopedic,79 min,General,"Stable, no allergies","Minimal pain, no complications",3,"Nausea, mild bleeding",1
3,4,65,F,26,Orthopedic,210 min,Local,"Stable, no allergies","Pain, slow recovery",7,Delayed recovery,1
4,5,65,M,28,Neurological,221 min,General,"Stable, no allergies","Pain, slow recovery",5,Respiratory distress,0


In [6]:
# Heart Disease — overview
print('=== HEART DISEASE UCI ===')
print()
print('Data types:')
print(heart.dtypes)
print()
print('Missing values (top 10):')
missing = heart.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0])
print(f'\nOverall missing: {heart.isnull().sum().sum() / (heart.shape[0]*heart.shape[1]) * 100:.1f}%')
print()
heart.head()

=== HEART DISEASE UCI ===

Data types:
id            int64
age           int64
sex          object
dataset      object
cp           object
trestbps    float64
chol        float64
fbs          object
restecg      object
thalch      float64
exang        object
oldpeak     float64
slope        object
ca          float64
thal         object
num           int64
dtype: object

Missing values (top 10):
ca          611
thal        486
slope       309
fbs          90
oldpeak      62
trestbps     59
thalch       55
exang        55
chol         30
restecg       2
dtype: int64

Overall missing: 11.9%



,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


### 🔍 Critical Discovery — Examine the Text Fields

Run the cells below and document what you find. How many unique values does each text column have?

In [7]:
# What's actually in the clinical notes?
print('=== PREOPERATIVE NOTES ===')
print(anes['PreoperativeNotes'].value_counts())
print(f'Unique values: {anes["PreoperativeNotes"].nunique()}')
print()
print('=== POSTOPERATIVE NOTES ===')
print(anes['PostoperativeNotes'].value_counts())
print(f'Unique values: {anes["PostoperativeNotes"].nunique()}')
print()
print('=== COMPLICATIONS ===')
print(anes['Complications'].value_counts())
print(f'Missing: {anes["Complications"].isnull().sum()} ({anes["Complications"].isnull().mean()*100:.1f}%)')

=== PREOPERATIVE NOTES ===
PreoperativeNotes
Hypertension, diabetes    157
Stable, no allergies      143
Name: count, dtype: int64
Unique values: 2

=== POSTOPERATIVE NOTES ===
PostoperativeNotes
Minimal pain, no complications    152
Pain, slow recovery               148
Name: count, dtype: int64
Unique values: 2

=== COMPLICATIONS ===
Complications
Respiratory distress     83
Nausea, mild bleeding    80
Delayed recovery         64
Name: count, dtype: int64
Missing: 73 (24.3%)


### ✏️ Your Observations

**What did you discover about the text columns?**

*Write your observations here:*
- PreoperativeNotes has only ___ unique values: ...
- PostoperativeNotes has only ___ unique values: ...
- This means that for NLP to be meaningful, we'll need to...


In [8]:
# Feature overlap between datasets
print('=== AGE OVERLAP ===')
print(f'Anesthesia age: {anes["Age"].min()} – {anes["Age"].max()}')
print(f'Heart age:      {heart["age"].min()} – {heart["age"].max()}')
print()

# Overlapping feature mapping
overlap_map = pd.DataFrame({
    'Anesthesia Feature': ['Age', 'BMI', 'PreoperativeNotes ("Hypertension")', 
                           'PreoperativeNotes ("diabetes")', 'SurgeryType=Cardiovascular'],
    'Heart Feature': ['age', '—', 'trestbps (blood pressure)', 
                      'fbs (fasting blood sugar)', 'num (disease severity)'],
    'Relationship': ['Direct match', 'No direct match in heart', 
                     'NLP-extracted hypertension ↔ structured BP',
                     'NLP-extracted diabetes ↔ structured glucose flag',
                     'NLP cardiac history ↔ heart disease diagnosis']
})
overlap_map

=== AGE OVERLAP ===
Anesthesia age: 33 – 72
Heart age:      28 – 77



,Anesthesia Feature,Heart Feature,Relationship
0,Age,age,Direct match
1,BMI,—,No direct match in heart
2,"PreoperativeNotes (""Hypertension"")",trestbps (blood pressure),NLP-extracted hypertension ↔ structured BP
3,"PreoperativeNotes (""diabetes"")",fbs (fasting blood sugar),NLP-extracted diabetes ↔ structured glucose flag
4,SurgeryType=Cardiovascular,num (disease severity),NLP cardiac history ↔ heart disease diagnosis


---
## Step 2 — Clean the Structured Data

Before we touch the text, let's get the structured data into shape.

### Anesthesia Dataset Cleaning

In [9]:
# 2a. Parse SurgeryDuration: '217 min' → 217
anes['duration_min'] = anes['SurgeryDuration'].str.extract(r'(\d+)').astype(int)

print('Before:', anes['SurgeryDuration'].head(3).tolist())
print('After: ', anes['duration_min'].head(3).tolist())
print(f'Duration range: {anes["duration_min"].min()} – {anes["duration_min"].max()} minutes')

Before: ['217 min', '181 min', '79 min']
After:  [217, 181, 79]
Duration range: 60 – 240 minutes


In [10]:
# 2b. Handle missing Complications
anes['has_complications'] = anes['Complications'].notna().astype(int)
anes['complication_type'] = anes['Complications'].fillna('None')

print(f'Complications missing: {anes["Complications"].isnull().sum()}')
print(f'has_complications distribution:')
print(anes['has_complications'].value_counts())

Complications missing: 73
has_complications distribution:
has_complications
1    227
0     73
Name: count, dtype: int64


In [11]:
# 2c. Encode categoricals
anes['gender_encoded'] = (anes['Gender'] == 'M').astype(int)
anes['anesthesia_general'] = (anes['AnesthesiaType'] == 'General').astype(int)

# Surgery type — one-hot encoding
surgery_dummies = pd.get_dummies(anes['SurgeryType'], prefix='surgery')
anes = pd.concat([anes, surgery_dummies], axis=1)

print('Encoded columns added:')
print(anes[['gender_encoded', 'anesthesia_general'] + list(surgery_dummies.columns)].head())

Encoded columns added:
   gender_encoded  anesthesia_general  surgery_Cardiovascular  \
0               1                   0                   False   
1               1                   0                    True   
2               0                   1                   False   
3               0                   0                   False   
4               1                   1                   False   

   surgery_Cosmetic  surgery_Neurological  surgery_Orthopedic  
0             False                  True               False  
1             False                 False               False  
2             False                 False                True  
3             False                 False                True  
4             False                  True               False  


### Heart Dataset Cleaning

In [12]:
# 2d. Fix problematic types in heart dataset
# fbs and exang are True/False STRINGS, not booleans
print('Before — fbs dtype:', heart['fbs'].dtype)
print('fbs unique:', heart['fbs'].unique())

heart['fbs'] = heart['fbs'].map({True: 1, False: 0, 'True': 1, 'False': 0})
heart['exang'] = heart['exang'].map({True: 1, False: 0, 'True': 1, 'False': 0})

print('After — fbs dtype:', heart['fbs'].dtype)
print('fbs values:', heart['fbs'].value_counts(dropna=False).to_dict())

Before — fbs dtype: object
fbs unique: [True False nan]
After — fbs dtype: float64
fbs values: {0.0: 692, 1.0: 138, nan: 90}


In [13]:
# 2e. Handle cholesterol zeros (biologically impossible = missing)
print(f'Cholesterol zeros: {(heart["chol"] == 0).sum()}')
heart['chol'] = heart['chol'].replace(0, np.nan)
print(f'Cholesterol missing after fix: {heart["chol"].isnull().sum()}')

Cholesterol zeros: 172
Cholesterol missing after fix: 202


In [14]:
# 2f. Impute missing values in heart dataset
heart_numeric_cols = ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
heart_cat_cols = ['fbs', 'exang', 'restecg', 'slope', 'thal']

# Numeric — median imputation
for col in heart_numeric_cols:
    median_val = heart[col].median()
    n_missing = heart[col].isnull().sum()
    heart[col] = heart[col].fillna(median_val)
    if n_missing > 0:
        print(f'{col}: filled {n_missing} missing with median {median_val:.1f}')

# Categorical — mode imputation
for col in heart_cat_cols:
    mode_val = heart[col].mode().iloc[0] if not heart[col].mode().empty else 'unknown'
    n_missing = heart[col].isnull().sum()
    heart[col] = heart[col].fillna(mode_val)
    if n_missing > 0:
        print(f'{col}: filled {n_missing} missing with mode "{mode_val}"')

trestbps: filled 59 missing with median 130.0
chol: filled 202 missing with median 239.5
thalch: filled 55 missing with median 140.0
oldpeak: filled 62 missing with median 0.5
ca: filled 611 missing with median 0.0
fbs: filled 90 missing with mode "0.0"
exang: filled 55 missing with mode "0.0"
restecg: filled 2 missing with mode "normal"
slope: filled 309 missing with mode "flat"
thal: filled 486 missing with mode "normal"


In [15]:
# 2g. Create binary heart disease target and encode remaining categoricals
heart['has_heart_disease'] = (heart['num'] > 0).astype(int)
print('Heart disease distribution:')
print(heart['has_heart_disease'].value_counts())

# Encode sex
heart['sex_encoded'] = (heart['sex'] == 'Male').astype(int)

# Encode remaining string columns
le_cols = ['cp', 'restecg', 'slope', 'thal']
label_encoders = {}
for col in le_cols:
    le = LabelEncoder()
    heart[col + '_encoded'] = le.fit_transform(heart[col].astype(str))
    label_encoders[col] = le

# Drop columns we won't need
heart = heart.drop(columns=['id', 'dataset'])

print(f'\nCleaned heart dataset: {heart.shape}')
print(f'Remaining missing: {heart.isnull().sum().sum()}')

Heart disease distribution:
has_heart_disease
1    509
0    411
Name: count, dtype: int64

Cleaned heart dataset: (920, 20)
Remaining missing: 0


### ✏️ Cleaning Summary

| Dataset | Action | Details |
|---------|--------|--------|
| Anesthesia | Parsed SurgeryDuration | String → integer |
| Anesthesia | Handled missing Complications | Created binary flag (73 NaN) |
| Anesthesia | Encoded categoricals | Gender, AnesthesiaType, SurgeryType |
| Heart | Fixed fbs/exang types | String True/False → binary int |
| Heart | Fixed cholesterol zeros | 0 → NaN → median imputed |
| Heart | Imputed missing values | Median for numeric, mode for categorical |
| Heart | Created binary target | num > 0 → has_heart_disease |

---
## Step 3 — Enrich the Clinical Notes

The raw notes have only 2 unique values each. That's too limited for meaningful NLP.

We'll build generators that produce **realistic, varied clinical notes** grounded in each patient's actual data. This teaches data augmentation and creates a proper NLP challenge.

**Key principle:** Because we generate the notes, we KNOW the ground truth — which gives us a way to validate our extraction later.

In [16]:
def generate_preop_note(row):
    """Generate a realistic preoperative note based on patient's structured data."""
    random.seed(row.name + 42)  # Reproducible per patient
    parts = []
    
    # --- Comorbidities based on original note ---
    if 'Hypertension' in row['PreoperativeNotes']:
        parts.append(random.choice([
            'History of hypertension',
            'HTN, managed with medication',
            'Elevated BP, on lisinopril 10mg',
            'Known hypertensive patient, BP controlled',
            'Chronic hypertension, on amlodipine',
            'HTN diagnosed 5 years ago'
        ]))
        parts.append(random.choice([
            'Type 2 diabetes mellitus',
            'DM controlled with metformin',
            'Diabetic, HbA1c 7.2',
            'History of diabetes, on oral hypoglycemics',
            'T2DM, diet controlled',
            'Diabetes mellitus, insulin dependent'
        ]))
    else:
        parts.append(random.choice([
            'No significant past medical history',
            'Generally healthy',
            'No chronic conditions noted',
            'Unremarkable medical history',
            'No prior hospitalizations',
            'Healthy, active lifestyle'
        ]))
    
    # --- BMI-based observations ---
    if row['BMI'] >= 30:
        parts.append(random.choice([
            'Obesity noted, BMI elevated',
            f'Obese, BMI {row["BMI"]}',
            'Weight management counseling provided',
            'Elevated BMI, increased surgical risk discussed'
        ]))
    elif row['BMI'] >= 25:
        if random.random() < 0.3:  # Only sometimes mentioned
            parts.append('Mildly overweight')
    
    # --- Age-based observations ---
    if row['Age'] >= 65:
        parts.append(random.choice([
            'Elderly patient, fall risk assessed',
            'Age over 65, cardiac clearance obtained',
            'Geriatric assessment complete',
            'Advanced age, additional monitoring planned'
        ]))
    elif row['Age'] >= 55:
        if random.random() < 0.3:
            parts.append('Middle-aged, routine screening current')
    
    # --- Surgery-specific ---
    if row['SurgeryType'] == 'Cardiovascular':
        parts.append(random.choice([
            'Cardiac history reviewed',
            'ECG shows normal sinus rhythm',
            'Pre-op stress test completed',
            'Echo shows preserved EF',
            'Cardiology clearance obtained'
        ]))
    elif row['SurgeryType'] == 'Orthopedic':
        if random.random() < 0.4:
            parts.append(random.choice([
                'Orthopedic pre-op checklist complete',
                'Musculoskeletal assessment done'
            ]))
    
    # --- Allergies ---
    parts.append(random.choice([
        'Allergies: NKDA',
        'No known drug allergies',
        'Allergies: Penicillin',
        'Allergies: sulfa drugs',
        'Allergies: latex',
        'NKDA per patient report'
    ]))
    
    # --- Lab results ---
    parts.append(random.choice([
        'Labs WNL',
        'CBC and BMP within normal limits',
        'Pre-op labs reviewed, no concerns',
        'Mild anemia noted on labs',
        'Labs unremarkable',
        'Hemoglobin 13.5, platelets adequate'
    ]))
    
    return '. '.join(parts) + '.'


# Apply to all patients
anes['enriched_preop'] = anes.apply(generate_preop_note, axis=1)

print(f'Unique enriched preop notes: {anes["enriched_preop"].nunique()}')
print(f'(Original had only {anes["PreoperativeNotes"].nunique()} unique values)')
print()
print('--- Sample enriched notes ---')
for i in [0, 1, 50, 100, 200]:
    print(f'\nPatient {i} (original: "{anes.iloc[i]["PreoperativeNotes"]}"):')
    print(f'  → {anes.iloc[i]["enriched_preop"]}')

Unique enriched preop notes: 298
(Original had only 2 unique values)

--- Sample enriched notes ---

Patient 0 (original: "Hypertension, diabetes"):
  → HTN diagnosed 5 years ago. Type 2 diabetes mellitus. Obesity noted, BMI elevated. NKDA per patient report. Pre-op labs reviewed, no concerns.

Patient 1 (original: "Stable, no allergies"):
  → No significant past medical history. Pre-op stress test completed. NKDA per patient report. CBC and BMP within normal limits.

Patient 50 (original: "Hypertension, diabetes"):
  → Known hypertensive patient, BP controlled. T2DM, diet controlled. Weight management counseling provided. Cardiac history reviewed. Allergies: latex. Labs unremarkable.

Patient 100 (original: "Hypertension, diabetes"):
  → Chronic hypertension, on amlodipine. Diabetes mellitus, insulin dependent. Elevated BMI, increased surgical risk discussed. No known drug allergies. Mild anemia noted on labs.

Patient 200 (original: "Stable, no allergies"):
  → No chronic conditions 

In [ ]:
def generate_postop_note(row):
    """Generate a realistic postoperative note based on patient's structured data."""
    random.seed(row.name + 99)  # Different seed from preop
    parts = []
    
    # --- Pain based on PostoperativeNotes + PainLevel ---
    if 'Pain, slow recovery' in row['PostoperativeNotes']:
        parts.append(random.choice([
            f'Patient reports pain level {row["PainLevel"]}/10',
            'Significant postoperative pain',
            'Pain requiring IV analgesics',
            'Moderate to severe pain at incision site',
            f'Pain score {row["PainLevel"]}, PCA initiated'
        ]))
        parts.append(random.choice([
            'Recovery slower than expected',
            'Delayed ambulation',
            'Extended PACU stay',
            'Patient requires additional monitoring',
            'Slow to return to baseline'
        ]))
    else:
        parts.append(random.choice([
            'Minimal postoperative pain',
            f'Pain well-controlled, level {row["PainLevel"]}/10',
            'Comfortable post-procedure',
            'Tolerating pain with oral medication',
            'Pain managed effectively'
        ]))
        parts.append(random.choice([
            'No complications observed',
            'Uncomplicated recovery',
            'Recovering as expected',
            'Stable in recovery room'
        ]))
    
    # --- Complications ---
    if pd.notna(row['Complications']):
        comp = row['Complications']
        if 'Respiratory' in comp:
            parts.append(random.choice([
                'Respiratory distress noted, O2 supplementation started',
                'Mild dyspnea post-extubation',
                'Respiratory complication, monitoring SpO2'
            ]))
        elif 'Nausea' in comp:
            parts.append(random.choice([
                'Nausea and vomiting, ondansetron given',
                'PONV managed with antiemetics',
                'Mild bleeding at surgical site, nausea reported'
            ]))
        elif 'Delayed' in comp:
            parts.append(random.choice([
                'Delayed recovery, extended observation',
                'Prolonged emergence from anesthesia',
                'Slower than expected return of reflexes'
            ]))
    
    # --- Vitals ---
    parts.append(random.choice([
        'Vitals stable',
        'Hemodynamically stable post-op',
        'BP and HR within normal limits',
        'Stable vital signs throughout recovery'
    ]))
    
    return '. '.join(parts) + '.'


anes['enriched_postop'] = anes.apply(generate_postop_note, axis=1)

print(f'Unique enriched postop notes: {anes["enriched_postop"].nunique()}')
print(f'(Original had only {anes["PostoperativeNotes"].nunique()} unique values)')
print()
print('--- Sample enriched postop notes ---')
for i in [0, 1, 50, 100, 200]:
    print(f'\nPatient {i} (original: "{anes.iloc[i]["PostoperativeNotes"]}", complications: {anes.iloc[i]["Complications"]}):')
    print(f'  → {anes.iloc[i]["enriched_postop"]}')

---
## Step 4 — Build the NLP Extraction Pipeline

Now we build a system that reads clinical notes and extracts structured features.

**Three components:** text preprocessing → regex-based entity extraction → feature columns.

In [ ]:
# 4a. Text preprocessing
def preprocess_note(note):
    """Clean clinical text for NLP extraction."""
    note = note.lower()
    note = re.sub(r'[^a-z0-9\s.,;/]', '', note)  # Keep clinical punctuation
    note = re.sub(r'\s+', ' ', note).strip()       # Normalize whitespace
    return note

anes['preop_clean'] = anes['enriched_preop'].apply(preprocess_note)
anes['postop_clean'] = anes['enriched_postop'].apply(preprocess_note)

# Show before/after
print('BEFORE:', anes.iloc[0]['enriched_preop'][:80], '...')
print('AFTER: ', anes.iloc[0]['preop_clean'][:80], '...')

In [ ]:
# 4b. Comorbidity extraction — regex patterns with clinical synonyms
#
# Each pattern matches multiple ways a condition might appear in clinical text.
# \b = word boundary (prevents 'cardiac' from matching inside 'myocardial')

comorbidity_patterns = {
    'hypertension': r'\b(hypertension|htn|elevated bp|hypertensive|high blood pressure|amlodipine|lisinopril)\b',
    'diabetes':     r'\b(diabetes|\bdm\b|diabetic|type 2|metformin|hba1c|hypoglycemic|insulin dependent|t2dm)\b',
    'obesity':      r'\b(obes[ei]|bmi elevated|elevated bmi|high bmi|overweight|weight management)\b',
    'cardiac':      r'\b(cardiac|ecg|echo|stress test|cardiology|sinus rhythm|preserved ef|lvh)\b',
    'anemia':       r'\b(anemia|anemic|low hemoglobin|low hgb)\b',
    'elderly_risk': r'\b(elderly|geriatric|fall risk|age over 65|advanced age)\b',
}

# Allergy detection — more complex because NKDA means NO allergies
allergy_positive_pattern = r'\ballergies?:\s*(penicillin|sulfa|latex)\b'
allergy_none_pattern = r'\b(nkda|no known drug allerg)\b'

print('Comorbidity patterns defined:', list(comorbidity_patterns.keys()))

In [ ]:
# 4c. Extraction function
def extract_comorbidities(note):
    """Extract clinical entities from a preprocessed note."""
    results = {}
    
    for condition, pattern in comorbidity_patterns.items():
        results[f'nlp_{condition}'] = int(bool(re.search(pattern, note, re.IGNORECASE)))
    
    # Allergy handling — check for specific allergies, but NOT NKDA
    has_specific_allergy = bool(re.search(allergy_positive_pattern, note, re.IGNORECASE))
    results['nlp_has_allergy'] = int(has_specific_allergy)
    
    return results


# Apply to preoperative notes
preop_features = anes['preop_clean'].apply(extract_comorbidities).apply(pd.Series)
anes = pd.concat([anes, preop_features], axis=1)

print('NLP features extracted:')
print(preop_features.sum().sort_values(ascending=False))

In [ ]:
# 4d. Postoperative note extraction
postop_patterns = {
    'nlp_pain':         r'\b(pain|analgesic|pca|hurts|discomfort|pain level \d)\b',
    'nlp_nausea':       r'\b(nausea|vomiting|ponv|antiemetic|ondansetron)\b',
    'nlp_respiratory':  r'\b(respiratory|dyspnea|o2|spo2|extubation|breathing)\b',
    'nlp_slow_recovery':r'\b(slow|delayed|prolonged|extended|longer than expected)\b',
    'nlp_stable':       r'\b(stable|uncomplicated|well.controlled|no complications|comfortable)\b',
}

def extract_postop(note):
    results = {}
    for name, pattern in postop_patterns.items():
        results[name] = int(bool(re.search(pattern, note, re.IGNORECASE)))
    return results

postop_features = anes['postop_clean'].apply(extract_postop).apply(pd.Series)
anes = pd.concat([anes, postop_features], axis=1)

print('Postop NLP features:')
print(postop_features.sum().sort_values(ascending=False))

In [ ]:
# 4e. Spot-check: show extraction results for sample patients
nlp_cols = [c for c in anes.columns if c.startswith('nlp_')]

print('=== SPOT CHECK: 5 Random Patients ===')
for idx in [0, 1, 42, 150, 250]:
    row = anes.iloc[idx]
    print(f'\n--- Patient {idx} ---')
    print(f'Preop note: "{row["preop_clean"][:80]}..."')
    extracted = {c: row[c] for c in nlp_cols if row[c] == 1}
    print(f'Extracted: {extracted if extracted else "(nothing found)"}')

---
## Step 5 — Validate Your NLP

Because we generated the notes, we know the ground truth. Let's measure extraction accuracy.

In [ ]:
# 5a. Build gold-standard labels from generation logic
# These mirror the EXACT conditions that triggered note content in Step 3

gold = pd.DataFrame(index=anes.index)
gold['hypertension'] = anes['PreoperativeNotes'].str.contains('Hypertension').astype(int)
gold['diabetes']     = anes['PreoperativeNotes'].str.contains('diabetes').astype(int)
gold['obesity']      = (anes['BMI'] >= 30).astype(int)
gold['cardiac']      = (anes['SurgeryType'] == 'Cardiovascular').astype(int)
gold['elderly_risk'] = (anes['Age'] >= 65).astype(int)

# Postop gold standards
gold['pain']          = anes['PostoperativeNotes'].str.contains('Pain, slow').astype(int)
gold['slow_recovery'] = anes['PostoperativeNotes'].str.contains('slow recovery').astype(int)
gold['has_complication'] = anes['has_complications']

print('Gold standard label counts:')
print(gold.sum())

In [ ]:
# 5b. Measure NLP accuracy per entity
print('=' * 60)
print('NLP EXTRACTION ACCURACY REPORT')
print('=' * 60)

validation_pairs = [
    ('hypertension', 'nlp_hypertension'),
    ('diabetes',     'nlp_diabetes'),
    ('obesity',      'nlp_obesity'),
    ('cardiac',      'nlp_cardiac'),
    ('elderly_risk', 'nlp_elderly_risk'),
]

accuracy_results = []
for gold_col, nlp_col in validation_pairs:
    print(f'\n--- {gold_col.upper()} ---')
    report = classification_report(gold[gold_col], anes[nlp_col], output_dict=True, zero_division=0)
    print(classification_report(gold[gold_col], anes[nlp_col], zero_division=0))
    
    accuracy_results.append({
        'Entity': gold_col,
        'Precision': report['1']['precision'],
        'Recall': report['1']['recall'],
        'F1': report['1']['f1-score'],
        'Support': report['1']['support']
    })

accuracy_df = pd.DataFrame(accuracy_results)
print('\n=== ACCURACY SUMMARY ===')
accuracy_df

In [ ]:
# 5c. Error analysis — find and examine failures
print('=== ERROR ANALYSIS ===')

for gold_col, nlp_col in validation_pairs:
    # False positives: NLP says yes, gold says no
    fp_mask = (anes[nlp_col] == 1) & (gold[gold_col] == 0)
    # False negatives: NLP says no, gold says yes
    fn_mask = (anes[nlp_col] == 0) & (gold[gold_col] == 1)
    
    fp_count = fp_mask.sum()
    fn_count = fn_mask.sum()
    
    if fp_count > 0 or fn_count > 0:
        print(f'\n--- {gold_col}: {fp_count} false positives, {fn_count} false negatives ---')
        if fp_count > 0:
            sample = anes.loc[fp_mask, 'preop_clean'].head(2)
            for s in sample:
                print(f'  FP: "{s[:100]}..."')
        if fn_count > 0:
            sample = anes.loc[fn_mask, 'preop_clean'].head(2)
            for s in sample:
                print(f'  FN: "{s[:100]}..."')

### ✏️ NLP Validation Notes

**Which entities had the best extraction accuracy? Why?**

*Your answer:*

**Which entities had errors? What caused the false positives/negatives?**

*Your answer:*

**What would you fix if you had another iteration?**

*Your answer:*

---
## Step 6 — Exploratory Data Analysis

Now that we have both structured and NLP-extracted features, let's explore patterns.

In [ ]:
# 6a. Anesthesia — distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(anes['Age'], bins=20, edgecolor='white', color='#2E75B6')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age')

axes[0,1].hist(anes['BMI'], bins=15, edgecolor='white', color='#548235')
axes[0,1].set_title('BMI Distribution')
axes[0,1].set_xlabel('BMI')

axes[1,0].hist(anes['duration_min'], bins=20, edgecolor='white', color='#C55A11')
axes[1,0].set_title('Surgery Duration')
axes[1,0].set_xlabel('Minutes')

axes[1,1].hist(anes['PainLevel'], bins=10, edgecolor='white', color='#7030A0')
axes[1,1].set_title('Pain Level Distribution')
axes[1,1].set_xlabel('Pain (0–10)')

plt.tight_layout()
plt.show()

In [ ]:
# 6b. Outcome by surgery type and NLP-extracted comorbidities
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Outcome by surgery type
ct1 = pd.crosstab(anes['SurgeryType'], anes['Outcome'], normalize='index')
ct1.plot(kind='bar', ax=axes[0], color=['#C55A11', '#2E75B6'])
axes[0].set_title('Outcome by Surgery Type')
axes[0].set_ylabel('Proportion')
axes[0].legend(['Outcome 0', 'Outcome 1'])
axes[0].tick_params(axis='x', rotation=45)

# Outcome by NLP-extracted hypertension
ct2 = pd.crosstab(anes['nlp_hypertension'], anes['Outcome'], normalize='index')
ct2.index = ['No Hypertension (NLP)', 'Hypertension (NLP)']
ct2.plot(kind='bar', ax=axes[1], color=['#C55A11', '#2E75B6'])
axes[1].set_title('Outcome by NLP-Extracted Hypertension')
axes[1].set_ylabel('Proportion')
axes[1].legend(['Outcome 0', 'Outcome 1'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# 6c. NLP feature prevalence
nlp_cols = [c for c in anes.columns if c.startswith('nlp_')]
prevalence = anes[nlp_cols].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#C55A11' if 'postop' in c or c in ['nlp_pain','nlp_nausea','nlp_respiratory','nlp_slow_recovery','nlp_stable'] 
          else '#2E75B6' for c in prevalence.index]
prevalence.plot(kind='barh', ax=ax, color=colors)
ax.set_title('NLP Feature Prevalence (Blue = Preop, Orange = Postop)')
ax.set_xlabel('Proportion of Patients')
plt.tight_layout()
plt.show()

In [ ]:
# 6d. Cross-dataset: age distribution comparison
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(anes['Age'], bins=20, alpha=0.6, label='Anesthesia', color='#2E75B6', edgecolor='white')
ax.hist(heart['age'], bins=20, alpha=0.6, label='Heart Disease', color='#C55A11', edgecolor='white')
ax.set_title('Age Distribution: Anesthesia vs. Heart Disease Datasets')
ax.set_xlabel('Age')
ax.legend()
plt.show()

print(f'Anesthesia age: {anes["Age"].min()}–{anes["Age"].max()} (mean {anes["Age"].mean():.0f})')
print(f'Heart age:      {heart["age"].min()}–{heart["age"].max()} (mean {heart["age"].mean():.0f})')
print(f'Overlap zone:   {max(anes["Age"].min(), heart["age"].min())}–{min(anes["Age"].max(), heart["age"].max())}')

In [ ]:
# 6e. Heart disease — correlation heatmap
heart_numeric = heart.select_dtypes(include=[np.number])
corr_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'has_heart_disease']
corr_cols = [c for c in corr_cols if c in heart_numeric.columns]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(heart_numeric[corr_cols].corr(), annot=True, cmap='RdBu_r', center=0, 
            fmt='.2f', ax=ax, square=True)
ax.set_title('Heart Disease — Feature Correlations')
plt.tight_layout()
plt.show()

### ✏️ EDA Observations

Write at least 5 observations:

1. 
2. 
3. 
4. 
5. 

---
## Step 7 — Merge Datasets & Feature Fusion

We'll merge on age bins and define three feature sets for comparison.

In [ ]:
# 7a. Create merge keys
age_bins = range(25, 80, 5)
anes['age_bin'] = pd.cut(anes['Age'], bins=age_bins)
heart['age_bin'] = pd.cut(heart['age'], bins=age_bins)

print('Anesthesia age bin distribution:')
print(anes['age_bin'].value_counts().sort_index())
print()
print('Heart age bin distribution:')
print(heart['age_bin'].value_counts().sort_index())

In [ ]:
# 7b. Merge on age bins
merged = pd.merge(
    anes, heart, 
    on='age_bin', 
    how='inner', 
    suffixes=('_surg', '_heart')
)

print(f'Anesthesia rows: {len(anes)}')
print(f'Heart rows:      {len(heart)}')
print(f'Merged rows:     {len(merged)}')
print(f'Merge expansion: {len(merged)/len(anes):.1f}x')
print(f'Merged columns:  {len(merged.columns)}')

In [ ]:
# 7c. Define three feature sets for comparison

# STRUCTURED ONLY — no NLP features
structured_features = [
    'Age', 'BMI', 'duration_min', 'PainLevel', 'gender_encoded', 'anesthesia_general',
    'has_complications',
    'surgery_Cardiovascular', 'surgery_Cosmetic', 'surgery_Neurological', 'surgery_Orthopedic',
]

# NLP ONLY — features extracted from text
nlp_features = [
    'nlp_hypertension', 'nlp_diabetes', 'nlp_obesity', 'nlp_cardiac',
    'nlp_anemia', 'nlp_elderly_risk', 'nlp_has_allergy',
    'nlp_pain', 'nlp_nausea', 'nlp_respiratory', 'nlp_slow_recovery', 'nlp_stable',
]

# FUSED — both structured and NLP
fused_features = structured_features + nlp_features

print(f'Structured features: {len(structured_features)}')
print(f'NLP features:        {len(nlp_features)}')
print(f'Fused features:      {len(fused_features)}')

# Verify all columns exist in the anesthesia dataframe (we'll model on anes, not merged)
for f in fused_features:
    assert f in anes.columns, f'Missing column: {f}'
print('\n✓ All feature columns verified.')

---
## Step 8 — Statistical Testing

Before modeling, let's test whether NLP-extracted features are associated with outcomes.

In [ ]:
# 8a. Hypothesis tests
print('=' * 60)
print('HYPOTHESIS TESTING')
print('=' * 60)

results = []

# Chi-square tests: NLP feature vs. Outcome
chi_pairs = [
    ('nlp_hypertension', 'Outcome', 'NLP Hypertension → Surgical Outcome'),
    ('nlp_diabetes', 'Outcome', 'NLP Diabetes → Surgical Outcome'),
    ('nlp_obesity', 'Outcome', 'NLP Obesity → Surgical Outcome'),
    ('nlp_cardiac', 'Outcome', 'NLP Cardiac History → Surgical Outcome'),
]

for feat, target, label in chi_pairs:
    ct = pd.crosstab(anes[feat], anes[target])
    chi2, p, dof, expected = chi2_contingency(ct)
    sig = '✓ Significant' if p < 0.05 else '✗ Not significant'
    print(f'\n{label}')
    print(f'  Chi-square = {chi2:.3f}, p = {p:.4f} → {sig}')
    results.append({'Test': label, 'Statistic': f'χ²={chi2:.3f}', 'p-value': p, 'Significant': p < 0.05})

# T-tests: PainLevel by comorbidity status
for feat, label in [('nlp_hypertension', 'Hypertension'), ('nlp_diabetes', 'Diabetes')]:
    group1 = anes.loc[anes[feat] == 1, 'PainLevel']
    group0 = anes.loc[anes[feat] == 0, 'PainLevel']
    t, p = ttest_ind(group1, group0)
    sig = '✓ Significant' if p < 0.05 else '✗ Not significant'
    print(f'\nPainLevel: {label} vs No {label}')
    print(f'  t = {t:.3f}, p = {p:.4f} → {sig}')
    print(f'  Mean pain (with): {group1.mean():.1f}, Mean pain (without): {group0.mean():.1f}')
    results.append({'Test': f'PainLevel by {label}', 'Statistic': f't={t:.3f}', 'p-value': p, 'Significant': p < 0.05})

print('\n' + '=' * 60)
results_df = pd.DataFrame(results)
results_df

---
## Step 9 — The Three-Way Model Comparison

This is the payoff. We train models on structured-only, NLP-only, and fused feature sets, then compare. **The key metric: does adding NLP features improve prediction?**

In [ ]:
# 9a. Prepare data
target = 'Outcome'
y = anes[target]

feature_sets = {
    'Structured': structured_features,
    'NLP-only':   nlp_features,
    'Fused':      fused_features,
}

# Train/test split (same split for all models)
X_full = anes[fused_features]
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Training set: {len(y_train)} samples')
print(f'Test set:     {len(y_test)} samples')
print(f'Target balance (train): {y_train.value_counts().to_dict()}')

In [ ]:
# 9b. Train all models and collect results
all_results = []
roc_data = {}  # For plotting

for fs_name, fs_cols in feature_sets.items():
    X_train = X_train_full[fs_cols]
    X_test = X_test_full[fs_cols]
    
    for model_name, model in [('LogReg', LogisticRegression(max_iter=1000, random_state=42)),
                               ('RandomForest', RandomForestClassifier(n_estimators=100, random_state=42))]:
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        auc = roc_auc_score(y_test, y_prob)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        
        label = f'{fs_name} — {model_name}'
        roc_data[label] = (fpr, tpr, auc)
        
        all_results.append({
            'Feature Set': fs_name,
            'Model': model_name,
            'Accuracy': report['accuracy'],
            'Precision': report['weighted avg']['precision'],
            'Recall': report['weighted avg']['recall'],
            'F1': report['weighted avg']['f1-score'],
            'AUC': auc,
        })
        
        # Save RF models for feature importance later
        if model_name == 'RandomForest' and fs_name == 'Fused':
            fused_rf = model
        if model_name == 'RandomForest' and fs_name == 'Structured':
            structured_rf = model

comparison_df = pd.DataFrame(all_results)
print('=== MODEL COMPARISON ===')
comparison_df

In [ ]:
# 9c. ROC Curve Comparison
fig, ax = plt.subplots(figsize=(10, 8))

colors = {
    'Structured': '#2E75B6',
    'NLP-only': '#C55A11',
    'Fused': '#548235',
}
linestyles = {'LogReg': '-', 'RandomForest': '--'}

for label, (fpr, tpr, auc) in roc_data.items():
    fs_name = label.split(' — ')[0]
    model_name = label.split(' — ')[1]
    ax.plot(fpr, tpr, 
            color=colors.get(fs_name, 'gray'),
            linestyle=linestyles.get(model_name, '-'),
            linewidth=2,
            label=f'{label} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves: Structured vs. NLP vs. Fused', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 9d. AUC Comparison Bar Chart
auc_summary = comparison_df.pivot(index='Feature Set', columns='Model', values='AUC')
auc_summary = auc_summary.reindex(['Structured', 'NLP-only', 'Fused'])

fig, ax = plt.subplots(figsize=(10, 6))
auc_summary.plot(kind='bar', ax=ax, color=['#2E75B6', '#548235'], edgecolor='white', width=0.7)
ax.set_title('AUC by Feature Set and Model', fontsize=14)
ax.set_ylabel('AUC Score')
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
ax.tick_params(axis='x', rotation=0)
ax.legend()

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=9)

plt.tight_layout()
plt.show()

# NLP Uplift
best_structured = comparison_df[comparison_df['Feature Set']=='Structured']['AUC'].max()
best_fused = comparison_df[comparison_df['Feature Set']=='Fused']['AUC'].max()
uplift = best_fused - best_structured
print(f'\n★ NLP UPLIFT: {uplift:+.4f} AUC')
print(f'  Best Structured AUC: {best_structured:.4f}')
print(f'  Best Fused AUC:      {best_fused:.4f}')

---
## Step 10 — Feature Importance & Clinical Interpretation

In [ ]:
# 10a. Feature importance from Fused Random Forest — color-coded by source
importances = fused_rf.feature_importances_
feat_imp = pd.DataFrame({
    'feature': fused_features,
    'importance': importances,
    'source': ['NLP' if f.startswith('nlp_') else 'Structured' for f in fused_features]
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = feat_imp['source'].map({'Structured': '#2E75B6', 'NLP': '#C55A11'})
ax.barh(feat_imp['feature'], feat_imp['importance'], color=colors)
ax.set_title('Feature Importance (Fused Random Forest)', fontsize=14)
ax.set_xlabel('Importance')

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#2E75B6', label='Structured'), 
                    Patch(color='#C55A11', label='NLP-Extracted')],
          loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

# How many NLP features in top 10?
top10 = feat_imp.tail(10)
nlp_in_top10 = (top10['source'] == 'NLP').sum()
print(f'\nNLP features in top 10: {nlp_in_top10} out of 10')

### ✏️ Interpretation

**Does NLP add value?** Quantify the uplift and explain what it means.

*Your answer:*

**Which NLP features contributed most?** Why do these make clinical sense?

*Your answer:*

**Synthetic data limitation:** Be honest about what this means for real-world applicability.

*Your answer:*

---
## Step 11 — NLP Pipeline Documentation & Error Analysis

### Pipeline Diagram

```
Raw Clinical Notes (2 unique values)
        │
        ▼
Note Enrichment (generate_preop_note / generate_postop_note)
        │  Uses: Age, BMI, SurgeryType, original note, Complications
        ▼
Enriched Notes (50+ unique variants)
        │
        ▼
Text Preprocessing (lowercase, clean punctuation, normalize whitespace)
        │
        ▼
Regex Entity Extraction (comorbidity_patterns + postop_patterns)
        │  Handles: synonyms, word boundaries, allergy negation
        ▼
NLP Feature Columns (binary flags: nlp_hypertension, nlp_diabetes, ...)
        │
        ▼
Validation (gold standard → precision / recall / F1 → error analysis → iterate)
        │
        ▼
Feature Fusion (structured_features + nlp_features → fused_features)
        │
        ▼
Three-Way Model Comparison → NLP Uplift Quantified
```

### Scalability Discussion

**Would this regex pipeline work on 10,000 real clinical notes?**

*Your answer:*

**What would a production NLP system need beyond regex?**

Consider: negation detection (NegEx), clinical NER models (MedSpaCy, Clinical BERT), abbreviation expansion, section detection (HPI vs. Assessment vs. Plan).

*Your answer:*

### Rule-Based vs. ML-Based NLP

| Aspect | Rule-Based (our regex) | ML-Based (spaCy / BERT) |
|--------|------------------------|-------------------------|
| Setup effort | | |
| Accuracy on known patterns | | |
| Handling new/unseen text | | |
| Negation handling | | |
| Maintenance | | |
| Speed | | |

*(Fill in the table with your assessment)*

---
## Step 12 — Conclusion & Limitations

### Key Findings

*(Fill in after completing the project)*

1. 
2. 
3. 
4. 
5. 

### Limitations

- The enriched notes were generated FROM structured data, so NLP features are partially derivative. In a real system with genuine clinical notes, NLP would capture information (medication dosages, family history, social history) that doesn't exist in any structured field.
- The dataset merge is approximate (age-binned, not patient-matched).
- The anesthesia dataset is small (300 patients).
- *(Add your own additional limitations)*

### Future Work

- 
- 
- 

---

*Go back to the top and fill in the Executive Summary now that you know your results.*